<a href="https://colab.research.google.com/github/xyt556/I-GUIDE-GeoAI-Education/blob/main/notebooks/12-vision-language-models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 视觉语言模型

## 简介

视觉语言模型 (VLM) 弥合了视觉感知和自然语言理解之间的鸿沟。与输出数字标签或边界框的传统计算机视觉模型不同，VLM 可以用通俗易懂的语言描述场景、回答有关图像内容的问题，并根据文本描述定位对象。

VLM 的发展建立在计算机视觉和自然语言处理的进步之上。2021 年，CLIP 在大规模对比预训练图像-文本对方面取得了突破，这表明经过训练以匹配图像和字幕的模型可以在数百个视觉任务上执行零样本分类，而无需进行特定任务的训练。

对于地理空间工作，VLM 实现了传统方法无法实现的任务。无需训练专门的分类器来检测游泳池，您可以简单地问 VLM“此图像中是否有任何游泳池？”这种零样本能力极大地降低了从遥感图像中提取信息的障碍。

VLM 有益于快速灾害损失评估、初步场地调查、环境监测和大规模土地利用清单等应用。分析师可以立即使用精心制作的提示从图像中提取见解，而无需等待自定义模型的训练。

在本教程中，您将通过 `geoai` 库中的 `MoondreamGeo` 类使用 [Moondream](https://moondream.ai)，这是一个针对边缘部署优化的轻量级开源 VLM。您还将使用 CLIP 对卫星图像进行文本引导分割。

## 学习目标

在本教程结束时，您将能够：

- 了解视觉语言模型如何结合视觉和文本推理
- 初始化和配置 `MoondreamGeo` 以进行可重现的地理空间分析
- 使用 Moondream 对卫星图像进行图像字幕和视觉问答
- 使用文本引导提示检测和定位地理参考图像中的对象
- 使用交互式 GUI 进行探索性 VLM 分析
- 实施滑动窗口方法，通过可配置的组合策略将 VLM 应用于大型栅格
- 对卫星图像执行基于 CLIP 的零样本分割
- 评估 VLM 在地球观测方面的优势和局限性

## 视觉语言模型的工作原理

视觉语言模型通过训练大量的图像-文本对来学习将图像与文本关联起来，将两种模态映射到共享的嵌入空间中，其中相关概念彼此靠近。有关更多详细信息，请参阅此 [VLM 解释器](https://huggingface.co/blog/vlms)。

**CLIP 和对比学习。** CLIP（对比语言-图像预训练）使用图像编码器和文本编码器，它们经过训练以最大化匹配图像-文本对之间的相似性，同时最小化不匹配对的相似性。在对数亿个网络来源的图像-文本对进行训练后，CLIP 形成了非常通用的视觉理解能力。

**从 CLIP 到生成式 VLM。** 生成式 VLM，如 Moondream，结合了视觉编码器（通常基于 CLIP 或 SigLIP）和语言模型解码器。视觉编码器将图像转换为特征标记，这些标记与文本标记一起馈送到语言模型中以生成自由格式的文本响应。

**Moondream 架构。** Moondream 将 SigLIP 视觉编码器与 Phi 语言模型配对，总计约 20 亿个参数。尽管其体积小巧，但它支持字幕、视觉问答、对象检测和点定位。

**为什么 VLM 对地理空间分析很重要。** 传统的遥感管道需要选择任务、收集标记数据、训练模型和部署它。VLM 将其折叠成一个灵活的接口，其中相同的模型可以通过不同的提示进行字幕、回答问题、检测对象和描述空间关系。

In [ ]:
# %pip install -U "geoai-py[extra]" transformers==4.57.6

Moondream v2 与 Transformers 5.x 不兼容，因此请安装版本 4.57.6 或其他兼容的 4.x 版本。

In [ ]:
import geoai
import leafmap
from geoai import MoondreamGeo

## 示例数据

下载包含建筑物、车辆和植被的停车场地理参考航拍图像。

In [ ]:
url = "https://data.source.coop/opengeos/geoai/parking-lot.tif"
image_path = geoai.download_file(url)

在交互式地图上可视化图像。

In [ ]:
m = leafmap.Map()
m.add_raster(image_path, layer_name="Satellite Image")
m

## 初始化 Moondream 处理器

`MoondreamGeo` 类封装了 Moondream 模型并提供地理空间支持。固定 `revision` 日期将模型权重锁定到特定的检查点以实现可重现性。首次运行将从 [Hugging Face](https://huggingface.co/vikhyatk/moondream2) 下载约 3.8 GB 的数据。

In [ ]:
processor = MoondreamGeo(
    model_name="vikhyatk/moondream2",
    revision="2025-06-21",
)

当使用 GeoTIFF 输入时，`MoondreamGeo` 会自动将像素坐标输出转换为地理坐标，并将结果作为 GeoDataFrames 返回，以便用于 GIS 工作流。

In [ ]:
result = processor.caption(image_path, length="short")
print(result["caption"])

```text
一张城市停车场的航拍图，展示了两座建筑物、车辆和树木，以及复杂的道路和停车位网络。
```

In [ ]:
result = processor.caption(image_path, length="normal")
print(result["caption"])

```text
航拍图展示了一个由仓库式建筑组成的综合体，大多呈浅灰色或白色，其中散布着几个停车场。一条宽阔的铺砌道路贯穿其中，有多条车道。图像左侧中心附近有一棵巨大的树，为工业景观增添了一抹绿色。停车场停满了汽车，表明该区域有活跃的人类活动。航拍视角突出了综合体复杂的布局。
```

In [ ]:
result = processor.caption(image_path, length="long")
print(result["caption"])

```text
这张航拍图显示了一个宽敞的停车场区域，周围环绕着两座大型建筑。停车场被清晰标记的停车位、道路和人行道分隔成多个部分。停车场停满了许多整齐排列的汽车，主要是白色和红色车辆。建筑物的屋顶是平坦的灰色，设计现代。该区域维护良好，停车场周围种植着绿树和灌木，为城市环境增添了一抹绿色。图像全面展示了停车场布局、停车状况和周围结构，清晰地展示了商业和住宅区。
```

`“long”` 字幕识别了较短字幕省略的特定功能。与任何生成模型一样，将字幕视为描述性摘要而不是事实真相。

In [ ]:
视觉问答 (VQA) 允许您提出关于图像的特定问题并获得文本答案，这对于评估土地覆盖、计算特征或评估状况很有用。

```text
图像中有两座建筑物。
```

In [ ]:
result = processor.query("What are the building roof colors?", image_path)
print(result["answer"])

```text
建筑屋顶颜色为白色和红色。
```

In [ ]:
result = processor.query(
    "What types of vehicles are visible in the parking areas?", image_path
)
print(result["answer"])

```text
停车场区域可见的车辆类型包括汽车、卡车和公共汽车。
```

VQA 非常适合快速评估任务，例如在自然灾害后查询航拍照片。具体、范围明确的问题比模糊的问题产生更可靠的答案，并且提供有关高度或比例的上下文有助于校准响应。

## 对象检测和点定位

Moondream 可以根据文本提示在图像中定位对象。`detect` 方法返回边界框，而 `point` 返回中心点坐标。两种方法都会自动将 GeoTIFF 输入的结果进行地理参考。

### 检测建筑物

In [ ]:
result = processor.detect(image_path, "building", output_path="buildings.geojson")
print(f"Detected {len(result['objects'])} buildings")

In [ ]:
result["gdf"]

将地理参考检测结果添加到地图。

In [ ]:
style = {"color": "red", "weight": 2}
m.add_gdf(result["gdf"], layer_name="Buildings", style=style)
m

### 定位建筑物质心

`point` 方法查找每个对象的中心点，这对于计算特征和分析空间分布很有用。

In [ ]:
result = processor.point(
    image_path, "building", output_path="building_centroids.geojson"
)
print(f"Found {len(result['points'])} building centroids")

In [ ]:
m.add_gdf(result["gdf"], layer_name="Building Centroids")
m

### 检测树木

对树木应用相同的检测方法进行定位。

In [ ]:
result = processor.detect(image_path, "tree", output_path="trees.geojson")
print(f"Detected {len(result['objects'])} trees")

In [ ]:
m.add_gdf(result["gdf"], layer_name="Trees", style={"color": "green", "weight": 2})

### 定位树木质心

In [ ]:
result = processor.point(image_path, "tree", output_path="tree_centroids.geojson")
print(f"Found {len(result['points'])} tree centroids")

添加树木质心以同时查看所有检测图层。

In [ ]:
m.add_gdf(result["gdf"], layer_name="Tree Centroids")
m

检测对于计算特征和识别空间分布很有用。对于精确描绘，可以考虑使用 VLM 检测作为分割模型（如 SAM）的初始提示。

In [ ]:
## 交互式 GUI

`MoondreamGeo` 类包含一个内置的交互式 GUI，用于探索性分析，无需为每个查询编写代码。

GUI 提供以下控件：

- **模式**：在字幕、查询、检测和点分析模式之间切换
- **提示**：输入查询、检测或点模式的文本提示（例如，“建筑物”、“绿树”或“停了多少辆车？”）
- **长度**：使用字幕模式时选择字幕长度（短、正常、长）
- **不透明度**：调整检测叠加层的不透明度
- **颜色**：选择边界框和点标记的颜色
- **运行**：执行选定的操作并在地图上显示结果
- **保存**：将结果导出到 GeoJSON 文件
- **重置**：清除所有结果并恢复原始地图视图

检测结果显示为边界框，点结果显示为地图上的圆形标记。运行操作后，您可以将结果作为 GeoDataFrame 访问。

In [ ]:
gdf = m_gui.last_result_as_gdf
gdf

GUI 对于探索图像、测试提示或向利益相关者展示 VLM 功能特别有用。

## 大型栅格的滑动窗口分析

卫星和航空图像通常太大，无法作为单个 VLM 输入进行处理。滑动窗口方法将栅格划分为更小的重叠瓦片，独立处理每个瓦片，并聚合结果。

关键参数包括：

- `window_size`：每个方形瓦片的像素尺寸（默认值：512）
- `overlap`：相邻瓦片之间重叠的像素数量（默认值：64）
- `iou_threshold`：合并重叠检测时进行非最大抑制的 IoU 阈值（默认值：0.5）
- `combine_strategy`：如何合并查询和字幕方法的每瓦片文本结果（`“concatenate”` 或 `“summarize”`）

### 使用滑动窗口进行对象检测

跨所有瓦片检测对象，并自动进行非最大抑制 (NMS) 以消除重叠区域中的重复项。

In [ ]:
result = processor.detect_sliding_window(
    image_path,
    "car",
    window_size=512,
    overlap=64,
    iou_threshold=0.5,
    output_path="cars_sliding_window.geojson",
)
print(f"Detected {len(result['objects'])} cars")

In [ ]:
result["gdf"].head()

在交互式地图上可视化滑动窗口汽车检测结果。

In [ ]:
m2 = leafmap.Map()
m2.add_raster(image_path, layer_name="Satellite Image")
m2.add_gdf(
    result["gdf"],
    layer_name="Detected Cars",
    style={"color": "red", "fillOpacity": 0.3},
)
m2

### 使用滑动窗口进行点检测

在大图像中查找特定对象位置作为点。

In [ ]:
trees = processor.point_sliding_window(
    image_path,
    "tree",
    window_size=512,
    overlap=64,
    output_path="trees_sliding_window.geojson",
)
print(f"Found {len(trees['points'])} tree locations")

在交互式地图上可视化检测到的树木位置。

In [ ]:
m3 = leafmap.Map()
m3.add_raster(image_path, layer_name="Satellite Image")
m3.add_gdf(trees["gdf"], layer_name="Trees", style={"color": "green", "radius": 3})
m3

### 使用滑动窗口进行视觉问答

逐瓦片查询大图像并组合每瓦片的答案。`“concatenate”` 策略将所有瓦片答案连接成一个字符串，保留所有信息，但可能读起来重复。

In [ ]:
result = processor.query_sliding_window(
    "What types of vehicles are visible?",
    image_path,
    window_size=512,
    overlap=64,
    combine_strategy="concatenate",
)
print(result["answer"])

```text
Tile 0 (region (0, 0, 512, 512)): The vehicles visible in the image include cars and trucks.
Tile 1 (region (448, 0, 960, 512)): The vehicles visible in the image include cars and trucks.
Tile 2 (region (896, 0, 1408, 512)): The vehicles visible in the image include cars and trucks.
Tile 3 (region (1344, 0, 1659, 512)): Cars are visible in the image.
Tile 4 (region (0, 448, 512, 938)): The vehicles visible in the image include cars and trucks.
Tile 5 (region (448, 448, 960, 938)): The vehicles visible in the image include cars and trucks.
Tile 6 (region (896, 448, 1408, 938)): The vehicles visible in the image include cars and trucks.
Tile 7 (region (1344, 448, 1659, 938)): Cars are visible in the image.
```

`“summarize”` 策略通过额外的模型调用将每瓦片答案提炼成一个连贯的响应，但代价是额外的计算。

In [ ]:
result = processor.query_sliding_window(
    "Describe the land use and features in this area.",
    image_path,
    window_size=512,
    overlap=64,
    combine_strategy="summarize",
)
print(result["answer"])

```text
The area in the image showcases a diverse range of land use and features, highlighting the adaptability and functionality of different urban environments. The parking lot, situated next to a building, demonstrates a well-organized system for managing the parking needs of vehicles. The parking lot is divided into multiple sections, with some areas having more open space than others, ensuring that vehicles can park efficiently and comfortably. The presence of trees surrounding the parking lot adds a touch of greenery and enhances the overall aesthetic appeal of the urban landscape. The parking lot appears to be well-maintained and organized, offering ample parking space for vehicles and accommodating various car types and sizes.
```

可以访问单个瓦片答案以进行空间分析。

In [ ]:
for tile in result["tile_answers"][:2]:  # Show first 2 tiles
    print(f"Tile {tile['tile_id']}: {tile['answer']}\n")

```text
瓦片 0：该区域有一个大型停车场，其中停放着许多汽车，排成行和列。停车场位于建筑物旁边，可能是一座商业或办公楼。停车场周围环绕着树木，为城市环境提供了自然元素。停车场看起来组织良好，有指定用于不同类型车辆的空间。

瓦片 1：图像中的区域似乎是一个停车场，有多个停车位，停满了各种汽车。停车场周围环绕着树木，为该区域营造了自然宜人的氛围。停车位按行和列排列，有些区域比其他区域有更多的开放空间。汽车以不同的角度和距离停放，占据了整个停车场。停车位的整体布局和排列表明，这是一个组织良好、高效的车辆停车系统。
```

### 使用滑动窗口进行图像字幕

通过对每个瓦片进行字幕并组合结果，为大型图像生成全面的字幕。

In [ ]:
result = processor.caption_sliding_window(
    image_path,
    window_size=512,
    overlap=64,
    length="normal",
    combine_strategy="concatenate",
)
print(result["caption"])

In [ ]:
result = processor.caption_sliding_window(
    image_path,
    window_size=512,
    overlap=64,
    length="long",
    combine_strategy="summarize",
)
print(result["caption"])

In [ ]:
geoai.empty_cache()

### 便利函数

对于不管理 `MoondreamGeo` 实例的一次性操作，geoai 库公开了模块级别的便利函数。

In [ ]:
from geoai import moondream_detect_sliding_window

result = moondream_detect_sliding_window(
    image_path,
    "car",
    window_size=512,
    overlap=64,
    model_name="vikhyatk/moondream2",
    revision="2025-06-21",
)
print(f"Detected {len(result['objects'])} cars")

In [ ]:
geoai.view_vector_interactive(result["gdf"], tiles=image_path)

### 比较常规检测与滑动窗口检测

一次性处理整个图像与逐瓦片处理通常会产生不同的检测计数，特别是对于小型或密集排列的对象。

In [ ]:
processor = MoondreamGeo(
    model_name="vikhyatk/moondream2",
    revision="2025-06-21",
)

In [ ]:
regular_result = processor.detect(image_path, "car")
print(f"Regular detection: {len(regular_result['objects'])} cars")

sliding_result = processor.detect_sliding_window(
    image_path, "car", window_size=512, overlap=64
)
print(f"Sliding window detection: {len(sliding_result['objects'])} cars")

```text
常规检测：50 辆车
滑动窗口检测：236 辆车
```

滑动窗口方法发现更多汽车，因为小型车辆在每个瓦片中占据更多像素。对于建筑物等大型对象，全图像推理通常就足够了，而滑动窗口对于车辆等小型特征更可靠。

In [ ]:
geoai.empty_cache()

### 性能提示

调整滑动窗口参数时，请考虑以下权衡：

- **窗口大小**：较小的瓦片（256-512 像素）可改善小型对象的检测，但会增加处理时间。较大的瓦片（512-1024 像素）速度更快，但可能会遗漏细节。
- **重叠**：较大的重叠（64-128 像素）可减少瓦片边界附近遗漏的对象，但会增加瓦片计数。较小的重叠（32-64 像素）速度更快，但可能会遗漏边界对象。
- **IoU 阈值**（仅检测）：较高的阈值（0.6-0.8）会保留更多检测结果，但可能会导致重复。较低的阈值（0.3-0.5）会激进地合并，并可能丢弃真正的邻近对象。
- **组合策略**：对于速度或每瓦片检查，使用 `“concatenate”`。当需要连贯的摘要时，使用 `“summarize”`。

In [ ]:
## 基于 CLIP 的分割

CLIPSeg 从文本提示生成分割掩码，无需边界框或点提示，生成一个软概率图，其中较高的值表示与文本描述更强的对齐。

在交互式地图上可视化卫星图像。

In [ ]:
geoai.view_raster(clip_image_path)

使用 512 像素的瓦片大小和 32 像素的重叠初始化 CLIPSeg 模型。

In [ ]:
segmenter = geoai.CLIPSegmentation(tile_size=512, overlap=32)

定义分割的输出路径和文本提示。

In [ ]:
mask_output_path = "tree_masks.tif"
text_prompt = "trees"

以 0.5 的概率阈值和高斯平滑运行分割。

In [ ]:
segmenter.segment_image(
    clip_image_path,
    output_path=mask_output_path,
    text_prompt=text_prompt,
    threshold=0.5,
    smoothing_sigma=1.0,
)

将生成的树木掩膜叠加在原始卫星图像上，并进行可视化。

In [ ]:
geoai.view_raster(
    mask_output_path,
    nodata=0,
    opacity=0.7,
    colormap="greens",
    layer_name="Trees",
    basemap=clip_image_path,
)

创建分割地图，将分割结果与原始图像进行比较。

In [ ]:
geoai.create_split_map(
    left_layer=mask_output_path,
    right_layer=clip_image_path,
    left_label="Trees",
    right_label="Satellite Image",
    left_args={"nodata": 0, "opacity": 0.8, "colormap": "greens"},
    basemap=clip_image_path,
)

当没有训练数据时，基于 CLIP 的分割对于快速土地覆盖制图很有用。软概率输出可以进行阈值处理以生成二值掩膜，或者跨多个提示组合以进行多类别制图，尽管结果通常不如监督方法准确。

## 地球观测中的实际应用

VLM 正在一系列地理空间领域中找到应用。

**灾害损失评估。** VLM 可以处理数千张灾后航拍图像，并提出“这座建筑的屋顶是否完好无损？”等问题，以优先考虑需要详细评估的区域。

**环境监测。** VLM 可以通过适当的提示协助识别非法采矿、检测森林砍伐、评估湿地健康和监测海岸侵蚀。

**城市分析。** 城市规划者可以使用 VLM 大规模表征城市环境，回答有关建筑密度、道路状况和绿地可用性等问题，涉及数千个图像瓦片。

**农业监测。** VLM 可以描述作物状况、识别田地边界和检测灌溉基础设施，而无需进行作物特定训练数据。

## 局限性和注意事项

**空间分辨率和比例。** VLM 可能难以处理卫星图像中的鸟瞰视角，尤其是在特征仅占据几个像素的高空。这是因为大多数 VLM 是在网络图像上训练的，这些图像通常以近乎水平的视角呈现。

**幻觉。** VLM 可能会产生自信但不正确的响应，描述不存在的特征或错误计数对象。务必根据参考数据验证输出。

**空间精度。** 边界框和点坐标是近似值，不应被视为测量级精度。

**一致性。** VLM 可能会在不同运行中为相同输入生成不同的输出。具有受限答案格式的结构化提示会产生更可靠的结果。

**提示敏感性。** 措辞的微小变化可能会产生截然不同的响应。开发标准化提示模板并根据事实真相进行验证对于操作工作流很重要。

**领域差距。** 大多数 VLM 是在网络图像而不是遥感数据集上训练的，因此性能可能因区域、季节和传感器类型而异。

## 主要收获

1. VLM 弥合了视觉感知和自然语言之间的鸿沟，从而实现了对地理空间图像的灵活零样本分析。
2. CLIP 通过对比学习学习共享的图像-文本表示，为许多 VLM 架构奠定了基础。
3. `MoondreamGeo` 封装了 Moondream 并提供地理空间支持，自动将输出地理参考为 GeoDataFrames。
4. Moondream 通过一致的 API 提供字幕、VQA、对象检测和点定位。
5. 交互式 GUI 支持探索性分析，无需编写额外的代码。
6. 滑动窗口方法通过可配置的组合策略将 VLM 功能扩展到任意大的栅格。
7. CLIPSeg 生成文本引导分割掩膜，用于快速土地覆盖制图，无需训练数据。
8. 由于潜在的幻觉、有限的空间精度和比例敏感性，应验证 VLM 输出。
9. 具体、结构化的提示比开放式查询产生更可靠的答案。